# 双向轮动：测量精度压力核查

只使用父Recorder与原生Qlib日历。固定0与半美分开盘压力，保留全部日期；不拟合、不读取未来收益。区间是给定测量情景下的保守包络，不是统计置信区间。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.analysis.rotation_measurement import float32_rounding_cell,rotation_measurement_bounds
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():
 if root==root.parent:raise RuntimeError('repository working directory required')
 root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/rotation_measurement_20261014';plan=json.loads((card/'plan.json').read_text());parent=root/plan['parent_card'];ps=json.loads((parent/'summary.json').read_text());mp=parent/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==plan['parent_manifest_sha256'];inventory=json.loads(mp.read_text());assert hashlib.sha256((root/plan['provider']/'qlib_store_manifest.json').read_bytes()).hexdigest()==plan['provider_manifest_sha256']
qlib.init(provider_uri=str(root/plan['provider']),region='us');calendar=pd.DatetimeIndex(D.calendar(start_time=plan['start'],end_time=plan['as_of']+' 16:00:00',freq='5min')).normalize().unique();assert len(calendar)==502
objects={}
with R.uri_context(ps['recorder_uri']):
 rec=R.get_recorder(recorder_id=ps['recorder_id'],experiment_name=parent.name)
 for name in ['inputs/weights','inputs/raw_sources','analysis/native','analysis/transitions']:
  e=next(e for e in inventory['entries'] if e['path'].endswith('/artifacts/'+name));p=Path(rec.download_artifact(name));assert hashlib.sha256(p.read_bytes()).hexdigest()==e['sha256'];objects[name]=rec.load_object(name)
weights=objects['inputs/weights'];native=objects['analysis/native'];prior=objects['analysis/transitions'].set_index('date');assert native.attrs.get('qlib_data_api')==['qlib.data.D.features']
# Source QA verifies that the rounding cells enclose actual stored original receipts.
checked=0;non_mill=0
for symbol,raw in objects['inputs/raw_sources'].items():
 raw=raw.loc[raw.time_key.str[:10].between(plan['start'],plan['as_of'])]
 for cutoff,field in [('09:35','open'),('09:45','close'),('10:30','close')]:
  part=raw.loc[raw.time_key.str[11:16].eq(cutoff)];observed=part[field].to_numpy(float);stamps=pd.to_datetime(part.time_key);values=native.xs(symbol)['$'+field].reindex(stamps).to_numpy(float);lo,hi=float32_rounding_cell(values);assert ((observed>=lo)&(observed<=hi)).all();checked+=len(observed);non_mill+=int((np.abs(observed*1000-np.rint(observed*1000))>1e-6).sum())
precision_audit={'endpoint_fields_checked':checked,'outside_float32_rounding_cell':0,'source_fields_not_exact_mill':non_mill,'round_to_mill_allowed':False}
x=native.reset_index();x['date']=x.datetime.dt.normalize();x['clock']=x.datetime.dt.strftime('%H:%M')
ep=weights.set_index(['instrument','datetime'])[['weight_pct']]
for cutoff,field,target in [('09:35','$open','open'),('09:45','$close','close_0945'),('10:30','$close','close_1030')]:
 v=x.loc[x.clock.eq(cutoff)].set_index(['instrument','date'])[field];v.index=v.index.set_names(['instrument','datetime']);ep=ep.join(v.rename(target))
rows=[]
for day,basket in ep.groupby(level='datetime'):
 basket=basket.copy();basket['weight']=basket.weight_pct/basket.weight_pct.sum();assert len(basket)==30
 for delta in plan['opening_halfwidths_usd']:
  known=bool(prior.loc[day,'known']);result=rotation_measurement_bounds(basket,opening_halfwidth=delta) if known else {k:np.nan for k in ['point','lower','upper','width','uncertain_member_weight']}
  if known:assert np.isclose(result['point'],prior.loc[day,'two_way_sign_rotation'],atol=1e-12,rtol=0)
  rows.append(dict(date=day,year=day.year,opening_halfwidth_usd=delta,known=known,**result))
bounds=pd.DataFrame(rows);assert len(bounds)==1004
stats=[]
for (year,delta),p in bounds.groupby(['year','opening_halfwidth_usd']):
 stats.append({'year':year,'opening_halfwidth_usd':delta,'calendar_sessions':len(p),'known_sessions':int(p.known.sum()),'point_positive_sessions':int(p.point.gt(0).sum()),'guaranteed_positive_sessions':int(p.lower.gt(0).sum()),'possibly_positive_sessions':int(p.upper.gt(0).sum()),'positive_point_not_guaranteed':int((p.point.gt(0)&p.lower.eq(0)).sum()),'zero_point_possible_positive':int((p.point.eq(0)&p.upper.gt(0)).sum()),'mean_point':float(p.point.mean()),'mean_lower':float(p.lower.mean()),'mean_upper':float(p.upper.mean()),'mean_width':float(p.width.mean()),'max_width':float(p.width.max())})
small=pd.DataFrame(stats);summary={'experiment':card.name,'decision':'measurement_bounds_only_no_information_qualification','new_fits':0,'new_backtests':0,'future_outcome_tests':0,'objective_achieved':False,'candidate_information_qualified':False,'provider_uri':str(root/plan['provider']),'region':'us','as_of':plan['as_of'],'parent_recorder_id':ps['recorder_id'],'precision_audit':precision_audit,'qlib_docs':['https://qlib.readthedocs.io/en/latest/component/data.html','https://qlib.readthedocs.io/en/latest/component/recorder.html'],'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='fixed_opening_precision_bounds',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,future_outcome_tests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,parent_inventory=inventory,endpoints=ep)
 rec.save_objects(artifact_path='analysis',bounds=bounds,annual_summary=small,precision_audit=precision_audit,summary=summary)
 rec.save_objects(artifact_path='sources',measurement_contract=(root/'src/analysis/rotation_measurement.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),endpoint_fields_checked=checked)
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');small.to_csv(card/'measurement_summary.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':summary['provider_uri'],'recorder_uri':summary['recorder_uri'],'artifact_policy':summary['artifact_policy']},sort_keys=False));assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(small.to_string(index=False));print(precision_audit)


# 双向轮动的测量敏感性：多数非零日保留，仍不是预测资格

本轮0拟合、0回测、0未来收益检验，沿用父卡2024—2025全部502个交易日、同一历史披露篮子、09:45/10:30切点和严格跨零公式。没有删除价格敏感日或选择更有利的epsilon。

先核对原始来源与Qlib存储：45930个端点字段全部落在原生float32舍入区间内；其中2812个原始字段并非0.001美元的整数倍，所以不能直接四舍五入到毫后再判断符号。正式描述仍来自父Qlib Recorder的原生价格输出。价格存储区间由相邻float32值的中点给出，不用自选数值容差改变收益符号。

固定两种情景：仅考虑三个端点的存储舍入；在此基础上，将每只股票的共同开盘基准上下扩展0.005美元，两个收盘端点仍仅保留存储区间。半美分来自父卡跨源开盘差异，不是按收益选出的阈值，也不是所有真实供应商误差的已证明上限。两个切点使用同一个开盘基准，不能分别挑开盘价制造跨越。

对转强/转弱权重分别计算必然与可能范围，给出2×min(U,D)的保守外包络。上界可能偏宽，尤其同一成员在端点舍入范围内有多种状态时；它不是紧界、概率区间或预测置信区间。

|年份|原非零日|半美分下仍确定非零|原非零变为不确定|原零值可能非零|平均区间宽度（权重百分点）|
|---|---|---|---|---|---|
|2024|124|123|1|10|0.5562|
|2025|144|143|1|10|0.7168|

合计268个原非零日中266个在该情景下仍确定非零；另有20个原零值日期可能非零。仅存储舍入时，原268个非零日全部保留，但12个原零值日存在可能非零状态。这说明不能把整个描述归为半美分噪声，也不能把所有零值当成确定没有双向轮动。

半美分情景平均包络宽度2024约0.5562、2025约0.7168个权重百分点，但最大分别16.0475、13.8743个百分点。因此“多数非零日保留”不等于数值大小、跨日排序或未来预测关系都稳定。保留完整逐日上下界，不筛掉不确定日期、不拿下界替换原公式，也不把剩余日期重新命名为独立样本外。

决定：measurement_bounds_only_no_information_qualification。候选仍为candidate_information_qualified=false，现役模型、恢复阈值和生产release未改，稳定净超额尚未实现。来源精度资格与预测增量、完整组合收益是三个不同结论。

下一项明确的信息问题是：较高双向轮动是否对应较低的实际10:32恢复动作净优势，而不只是重复ETF早盘涨跌/波动。应使用既有同状态beta=0.6、10万美元Tiered、2/5bp独立路径的恢复/HOLD局部对照标签，在检验前冻结预期方向、共同市场控制、主指标和失败停止规则；该局部标签也不等于完整策略价值。不能根据这一测量结果直接调仓、换标签期限或搜索阈值。

验证：一个集中合同测试枚举成员级端点情景并确认包络覆盖、压力范围扩大会扩大包络、纯单向修复仍为零；1 passed。执行notebook并核对父Recorder与provider哈希。NAS冷读回记录另见readback_verification.json。

[年度测量结果](measurement_summary.csv) · [固定方案](plan.json)

Recorder `fd69ab9a4167431292f6ed1d08e82e90`。
